# Fan-Sim Colab Smoke Run

This notebook is for Colab smoke tests: install Fan-Sim, optionally install OpenFOAM 2406, run one short case, export VTU, build one graph, and train a small model. Full OpenFOAM sweeps should run on a Linux SSH server.

In [ ]:
# Optional: mount Google Drive for persistent storage.
from google.colab import drive
drive.mount('/content/drive')

Clone or pull the project from GitHub. Keep OpenFOAM input data and generated artifacts outside git; copy the base case into `data/base_case` after the repo is ready.

In [ ]:
REPO_URL = 'https://github.com/tcthuong/fan-sim.git'
PROJECT_ROOT = '/content/fan-sim'
import os

if os.path.isdir(f'{PROJECT_ROOT}/.git'):
    !git -C "$PROJECT_ROOT" pull --ff-only
else:
    !rm -rf "$PROJECT_ROOT"
    !git clone "$REPO_URL" "$PROJECT_ROOT"

%cd $PROJECT_ROOT
!pwd
!python --version

In [ ]:
# Smoke install. Use this first because it avoids the large PhysicsNeMo/Torch install.
!python -m pip install --upgrade pip
!python -m pip install -e ".[service,vtk,dev]"
!python -m pytest -q
!fan-sim --help

In [ ]:
# Optional heavy ML install for PhysicsNeMo MeshGraphNet.
# Skip this if you only want to run OpenFOAM/export/build-graphs or numpy smoke training.
# !python -m pip install -e ".[ml]"
# !python -c "from physicsnemo.models.meshgraphnet.meshgraphnet import MeshGraphNet; print('MeshGraphNet ok')"

In [ ]:
# Install OpenFOAM 2406. This can take several minutes and can fail if Colab changes its Ubuntu image.
!bash scripts/colab_install_openfoam2406.sh

In [ ]:
# Generate the one-case Colab smoke matrix.
!fan-sim generate-cases --config configs/fan_sim_colab.yaml

In [ ]:
# Patch generated controlDict to a two-iteration smoke run.
from pathlib import Path
p = Path('runs/openfoam/case_rpm_0600_pout_000/system/controlDict')
s = p.read_text()
s = s.replace('endTime 1000.0;', 'endTime 2;')
s = s.replace('writeInterval 1000;', 'writeInterval 1;')
p.write_text(s)
print(p)

In [ ]:
!fan-sim run-openfoam --config configs/fan_sim_colab.yaml --case-id case_rpm_0600_pout_000
!tail -120 runs/openfoam/case_rpm_0600_pout_000/log.fan-sim-openfoam

In [ ]:
!fan-sim export-vtk --config configs/fan_sim_colab.yaml --case-id case_rpm_0600_pout_000
!find runs/openfoam/case_rpm_0600_pout_000/VTK -name '*.vtu' -maxdepth 3 -print

In [ ]:
!fan-sim build-graphs --config configs/fan_sim_colab.yaml
!find artifacts/graphs -name '*.graph.pt' -print

In [ ]:
# configs/fan_sim_colab.yaml uses model.backend: numpy by default.
!fan-sim train --config configs/fan_sim_colab.yaml --epochs 1
!find artifacts/models/fan_mgn_colab -maxdepth 2 -type f -print